# EME stability settings and one-axis neff interpolation

This notebook uses the bundled Si waveguide dataset to show how to pass
``EMEStabilityConfig`` to ``EME`` and enable optional phase interpolation.
The default thresholds were selected from SiN convergence tests at 1.55 µm;
the Si dataset here demonstrates the API, not a new threshold calibration.

Run the notebook from the repository root or the ``examples/`` directory.


In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

import numpy as np

repository_root = Path.cwd()
if not (repository_root / "em_simulation").is_dir():
    repository_root = repository_root.parent
if not (repository_root / "em_simulation").is_dir():
    raise FileNotFoundError("Run this notebook from the repository root or examples/.")

sys.path.insert(0, str(repository_root))
import em_simulation as sim

dataset_path = repository_root / "sample_datasets" / "Si_rectangular_single_waveguide"
dataset = sim.DataUpdater(str(dataset_path), is_testmode=True)
geometry = sim.LinearTaper(dataset, 1e-6, 2e-6, 10e-6)
launch = [1, 0, 0, 0, 0]
default_config = sim.EMEStabilityConfig()

print(default_config)


## Compare propagation phases

Both simulations below use the *same* stability configuration. The only
difference is ``neff_interpolation=True``. It interpolates guided-mode
effective indices along the taper's varying width when a tracked pair of
neighboring dataset modes is available. Interface overlaps and PML decisions
remain based on the original dataset.


In [ ]:
baseline_eme = sim.EME(geometry, stability_config=default_config)
baseline_result = sim.Runner(baseline_eme).propagate(launch)

interpolated_eme = sim.EME(
    geometry,
    stability_config=default_config,
    neff_interpolation=True,
)
interpolated_result = sim.Runner(interpolated_eme).propagate(launch)

print("Interpolation diagnostics:", interpolated_eme.neff_interpolation_diagnostics)
print(
    "Final-section amplitude difference norm:",
    np.linalg.norm(interpolated_result[-1] - baseline_result[-1]),
)


The diagnostic ``axis`` should be ``top_width`` for this taper.
``interpolated_mode_steps`` counts the guided mode-steps that received an
interpolated phase. A guided mode-step without an unambiguous pair keeps its
original phase.


## Change one stability setting

``pml_sink_loss`` controls when a present mode's phase transmission is set
to zero in direct S-matrix propagation. It is independent of
``pml_regularization_loss``, which affects the reliability weights used
in interface and feedback solves. The value below is only an API example;
check convergence before using it as a physical setting.


In [ ]:
custom_config = replace(default_config, pml_sink_loss=1.0e-2)
custom_eme = sim.EME(geometry, stability_config=custom_config)
custom_result = sim.Runner(custom_eme).propagate(launch)

print("Custom sink threshold:", custom_config.pml_sink_loss)
print(
    "Final-section amplitude difference from default configuration:",
    np.linalg.norm(custom_result[-1] - baseline_result[-1]),
)


## Transfer-matrix path

The default PML sink is supported by direct S-matrix propagation. A
zero-transmission sink phase cannot be inverted for a transfer matrix, so
disable the sink if you explicitly call ``calc_Tmatrix()``.


In [ ]:
transfer_config = replace(default_config, pml_mode_sink=False)
transfer_eme = sim.EME(geometry, stability_config=transfer_config)
transfer_eme.calc_Tmatrix()
print("Transfer matrices calculated with PML sink disabled.")


## Scope and diagnostics

``neff_interpolation`` is off by default and supports at most one varying
dataset parameter. It raises an error for simultaneous variation of two
or more parameters or a physical profile outside the dataset grid.
Interpolation changes the propagation phase only; it does not interpolate
mode fields, change interface overlaps, or change stability weights.

See the [EME options guide](https://dataset-based-emedreme.readthedocs.io/en/latest/EME_stability_and_interpolation.html)
for the interface SVD, feedback, and PML sink equations.
